<div id="singlestore-header" style="display: flex; background-color: #525252; padding: 5px;">
    <div id="icon-image" style="width: 90px; height: 90px;">
        <img width="100%" height="100%" src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/header-icons/brand/pipeline.png" />
    </div>
    <div id="text" style="padding: 5px; margin-left: 10px;">
        <div id="badge" style="display: inline-block; background-color: rgba(0, 0, 0, 0.15); border-radius: 4px; padding: 4px 8px; align-items: center; margin-top: 6px; margin-bottom: -2px; font-size: 80%; color: #FFFFFF;">SingleStore Notebooks</div>
        <h1 style="font-weight: 500; margin: 8px 0 0 4px; color: #FFFFFF;">Ingest and Operate Modern Data Pipelines</h1>
    </div>
</div>

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Note</b></p>
        <p>This notebook requires a Standard or Premium cluster. Pipelines are not available on Free Starter clusters.</p>
    </div>
</div>

## What you will learn

This notebook covers four pipeline operations in SingleStore:

1. Load purchase events from S3 with `CREATE PIPELINE`
2. Update a live pipeline without losing progress using `ALTER PIPELINE`
3. Profile ingestion throughput with `PIPELINES_BATCHES`
4. Replace polling with event-driven Kinesis file discovery

**Dataset:** Real-Time Digital Marketing purchase events (public S3, no credentials needed).

---

## Before you start

**Cluster requirement:** Standard or Premium. Pipelines are not available on Free Starter clusters.

**Setup steps:**
1. Select your cluster and any database from the dropdowns at the top.
2. Run cells top to bottom. To rerun from scratch, run the **Section 7 — Cleanup** cell first.

> After Section 1, select `pipelines_demo` from the database dropdown so `%%sql` DDL lands in the right database.

## How the pieces fit together

A **pipeline** continuously reads files from S3 (or Kafka) and loads them into a table, tracking which files have already been processed so restarts pick up where they left off.

```
S3 bucket (NDJSON / Parquet files)
          │
          ▼  CREATE LINK → CREATE PIPELINE
   SingleStore pipeline
          │   ├── ALTER PIPELINE  (update config without resetting offsets)
          │   ├── PIPELINES_BATCHES  (per-batch throughput metrics)
          │   └── Kinesis notification  (event-driven file discovery)
          ▼
    target table
```

## Section 1 — Connect and create the working database

Installs dependencies, connects to your cluster, and creates `pipelines_demo`.

**Two SQL patterns used in this notebook:**
- **`%%sql`** — DDL, control statements, and display SELECTs.
- **`run_sql()`** — where Python is needed: error handling, polling loops, and DataFrame results.

After running the cell below, select **`pipelines_demo`** from the database dropdown above.

In [1]:
%pip install singlestoredb pandas --quiet

In [2]:
import time
import pandas as pd
import singlestoredb as s2

pd.set_option("display.max_colwidth", 120)

DB_NAME = "pipelines_demo"

def connect():
    global conn
    conn = s2.connect()
    conn.autocommit(True)

def run_sql(query, params=None):
    for attempt in range(4):
        try:
            with conn.cursor() as cur:
                cur.execute(query, params)
                if cur.description:
                    cols = [d[0] for d in cur.description]
                    return pd.DataFrame(cur.fetchall(), columns=cols)
            return None
        except Exception as e:
            msg = str(e)
            if getattr(e, "errno", None) in (2006, 2013) or "Lost connection" in msg or "gone away" in msg:
                connect()
            else:
                raise

connect()
print("SingleStore version:", run_sql("SELECT @@memsql_version AS v").iloc[0, 0])
print()
print("Next: select 'pipelines_demo' from the database dropdown above, then run the next cell.")

In [3]:
%%sql
CREATE DATABASE IF NOT EXISTS pipelines_demo;

In [4]:
# Point the run_sql() connection at pipelines_demo so unqualified table/pipeline
# references in Python cells resolve to the same database as %%sql DDL.
run_sql(f"USE {DB_NAME}")
print("Working database:", run_sql("SELECT DATABASE() AS db").iloc[0, 0])

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Action Required</b></p>
        <p>Select <tt style="color: inherit; background-color: rgba(0,0,0,0.08);">pipelines_demo</tt> from the database dropdown at the top of this notebook before running any cells below. This ensures all <code style="color: inherit; background-color: rgba(0,0,0,0.08);">%%sql</code> DDL lands in the correct database.</p>
    </div>
</div>

## Section 2 — Create a table and load S3 data with a pipeline

Creates `rtdm_purchases_raw`, points a pipeline at a public S3 bucket, and waits for the first batch. Also inserts sample rows with a JSON `tags` column to show field extraction from complex types.

In [5]:
# Stop and drop any pipelines from a previous run before touching the table.
# A pipeline holds a reference to its target table — DROP TABLE will fail
# if the pipeline still exists.
for p in ["rtdm_pipeline", "rtdm_pipeline_glob"]:
    try:
        run_sql(f"STOP PIPELINE {p}")
    except Exception:
        pass
    run_sql(f"DROP PIPELINE IF EXISTS {p}")
print("Pipelines cleared. Ready to create the target table.")

In [6]:
%%sql
DROP TABLE IF EXISTS rtdm_purchases_raw;

CREATE TABLE rtdm_purchases_raw (
    seq_id        BIGINT,
    subscriber_id BIGINT,
    vendor        VARCHAR(200),
    tags          JSON,
    ts            DATETIME(6) DEFAULT NOW(6),
    SORT KEY (ts)
);

### Create and start the pipeline

The cell below creates the pipeline (reading NDJSON from public S3, no credentials needed), starts it, and polls until the first batch arrives (up to ~60 seconds). `BATCH_INTERVAL 2500` commits a batch every 2.5 seconds.

In [7]:
%%sql
DROP PIPELINE IF EXISTS rtdm_pipeline;

CREATE PIPELINE rtdm_pipeline
AS LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1k-2p/purchases.*.json'
CONFIG '{"region": "us-east-1"}'
BATCH_INTERVAL 2500
INTO TABLE rtdm_purchases_raw
FORMAT JSON
(
    seq_id        <- seqId,
    subscriber_id <- subscriberid,
    @v            <- vendor
)
SET vendor = FROM_BASE64(@v);

START PIPELINE rtdm_pipeline;

In [8]:
print("Pipeline started. Waiting for first batch...")
for _ in range(12):
    time.sleep(5)
    n = run_sql("SELECT COUNT(*) AS n FROM rtdm_purchases_raw").iloc[0, 0]
    if int(n) > 0:
        print(f"Rows loaded: {n}")
        break
    print("  ...waiting")

In [9]:
%%sql
SELECT seq_id, subscriber_id, vendor, ts
FROM   rtdm_purchases_raw
LIMIT  5;

### Extracting fields from JSON columns

The `tags` column lands as raw JSON. The cell below inserts sample rows then uses `JSON_EXTRACT_STRING` to pull out individual fields — the same pattern applies to Parquet map and array columns.

In [10]:
%%sql
DELETE FROM rtdm_purchases_raw WHERE seq_id >= 999001;

INSERT INTO rtdm_purchases_raw (seq_id, subscriber_id, vendor, tags)
VALUES
    (999001, 1, 'DemoVendorA', '{"category": "new_visitor", "device": "mobile", "time_of_day": "afternoon"}'),
    (999002, 2, 'DemoVendorB', '{"category": "new_visitor", "device": "mobile", "time_of_day": "afternoon"}'),
    (999003, 3, 'DemoVendorC', '{"category": "new_visitor", "device": "mobile", "time_of_day": "afternoon"}');

In [11]:
%%sql
SELECT seq_id, vendor,
       JSON_EXTRACT_STRING(tags, 'category')    AS category,
       JSON_EXTRACT_STRING(tags, 'device')      AS device,
       JSON_EXTRACT_STRING(tags, 'time_of_day') AS time_of_day
FROM   rtdm_purchases_raw
WHERE  seq_id >= 999001;

<div class="alert alert-block alert-info">
    <b class="fa fa-solid fa-info-circle"></b>
    <div>
        <p><b>Parquet type mapping</b></p>
        <p>Map Parquet complex fields in the column list of <code style="color: inherit; background-color: rgba(0,0,0,0.08);">CREATE PIPELINE</code>. Arrays and maps land as <code style="color: inherit; background-color: rgba(0,0,0,0.08);">JSON</code>; use <code style="color: inherit; background-color: rgba(0,0,0,0.08);">JSON_EXTRACT_STRING</code>, <code style="color: inherit; background-color: rgba(0,0,0,0.08);">JSON_EXTRACT_DOUBLE</code>, or <code style="color: inherit; background-color: rgba(0,0,0,0.08);">JSON_ARRAY_UNPACK</code> to access individual elements.</p>
    </div>
</div>

## Section 3 — Update a live pipeline with `ALTER PIPELINE`

Dropping and recreating a pipeline resets its file offset, causing every file to be re-ingested. `ALTER PIPELINE` updates the connection config (bucket path, credentials, region) **without resetting offsets**.

**Pattern:** stop → alter → restart.

The cell below simulates a config update after a bucket rename or credential rotation, then confirms the change in `information_schema.PIPELINES`.

In [12]:
%%sql
-- Always stop before altering to avoid unpredictable behaviour.
STOP PIPELINE rtdm_pipeline;

ALTER PIPELINE rtdm_pipeline
SET CONFIG '{"region": "us-east-1", "disable_gunzip": false}';

START PIPELINE rtdm_pipeline;

In [13]:
%%sql
SELECT PIPELINE_NAME, STATE, ALTER_TIME, CONFIG_JSON
FROM   information_schema.PIPELINES
WHERE  PIPELINE_NAME = 'rtdm_pipeline';

<div class="alert alert-block alert-warning">
    <b class="fa fa-solid fa-exclamation-circle"></b>
    <div>
        <p><b>Always stop before altering</b></p>
        <p>Run <code style="color: inherit; background-color: rgba(0,0,0,0.08);">STOP PIPELINE</code> before <code style="color: inherit; background-color: rgba(0,0,0,0.08);">ALTER PIPELINE</code>. Altering a running pipeline can cause unpredictable behaviour. Restart with <code style="color: inherit; background-color: rgba(0,0,0,0.08);">START PIPELINE</code> after the change is applied.</p>
    </div>
</div>

## Section 4 — Profile ingestion with `PIPELINES_BATCHES`

`information_schema.PIPELINES_BATCHES` records one row per partition per completed batch, giving you timing and throughput metrics to diagnose slow ingestion.

### Key columns

<table>
  <tr><th>Column</th><th>What it measures</th></tr>
  <tr><td><code>BATCH_TIME</code></td><td>Wall-clock seconds for the batch</td></tr>
  <tr><td><code>BATCH_ROWS_INSERTED</code></td><td>Rows successfully committed</td></tr>
  <tr><td><code>BATCH_ROWS_WRITTEN</code></td><td>Total rows written (inserted + updated)</td></tr>
</table>

High `BATCH_TIME` relative to `BATCH_ROWS_INSERTED` indicates a parse or commit bottleneck. `BATCH_ROWS_WRITTEN` much higher than `BATCH_ROWS_INSERTED` means rows are being skipped or reprocessed.

In [14]:
%%sql
SELECT BATCH_ID, BATCH_STATE,
       FROM_UNIXTIME(MIN(BATCH_START_UNIX_TIMESTAMP)) AS batch_start,
       MAX(BATCH_TIME)           AS elapsed_sec,
       SUM(BATCH_ROWS_INSERTED)  AS rows_inserted,
       SUM(BATCH_ROWS_WRITTEN)   AS rows_written
FROM   information_schema.PIPELINES_BATCHES
WHERE  PIPELINE_NAME = 'rtdm_pipeline'
GROUP  BY BATCH_ID, BATCH_STATE
ORDER  BY BATCH_ID DESC
LIMIT  10;

## Section 5 — Event-driven S3 file discovery via Kinesis

By default, SingleStore polls S3 on a fixed interval. **Kinesis-based event-driven discovery** eliminates polling: S3 publishes an `ObjectCreated` event the moment a file lands, and SingleStore begins downloading immediately.

### Setup

1. In AWS: create an S3 event notification sending `s3:ObjectCreated:*` to a Kinesis Data Stream.
2. In SingleStore, add `KINESIS_STREAM` to your pipeline:

```sql
CREATE PIPELINE rtdm_pipeline_kinesis
AS LOAD DATA S3 'your-bucket/prefix/*'
CONFIG '{"region": "us-east-1"}'
KINESIS_STREAM 'arn:aws:kinesis:us-east-1:123456789:stream/your-stream'
CREDENTIALS '{"aws_access_key_id": "...", "aws_secret_access_key": "..."}'
BATCH_INTERVAL 2500
INTO TABLE rtdm_purchases_raw
FORMAT JSON (...);
```

### Polling vs. event-driven

<table>
  <tr><th></th><th>Polling</th><th>Kinesis event-driven</th></tr>
  <tr><td>Latency</td><td>Seconds to minutes</td><td>Near real-time</td></tr>
  <tr><td>S3 API cost</td><td>List calls every interval</td><td>Event notifications only</td></tr>
  <tr><td>Setup</td><td>None</td><td>S3 notification + Kinesis stream</td></tr>
  <tr><td>Best for</td><td>Batch / hourly loads</td><td>Low-latency streaming ingest</td></tr>
</table>

> **Note:** This section is reference-only — AWS credentials and a configured stream are required to run.

## Section 6 — Target specific files with glob patterns

More specific glob patterns reduce the number of files evaluated on each discovery scan, directly cutting discovery time.

### Supported glob syntax

<table>
  <tr><th>Pattern</th><th>Matches</th></tr>
  <tr><td><code>*</code></td><td>Any string (existing behaviour)</td></tr>
  <tr><td><code>?</code></td><td>Any single character</td></tr>
  <tr><td><code>[abc]</code></td><td>One of: a, b, or c</td></tr>
  <tr><td><code>[a-z]</code></td><td>Any character in the range</td></tr>
  <tr><td><code>[^abc]</code></td><td>Any character <em>not</em> in the set</td></tr>
</table>

### Examples
```sql
-- Only October dates 01–29
'my-bucket/purchases/2026-10-[0-2]?.json'
-- Single-character vendor prefix
'my-bucket/events/vendor_?.parquet'
-- Exclude staging files starting with underscore
'my-bucket/events/[^_]*.json'
```

The pattern `000[0-2]` matches files ending in `.0000`, `.0001`, or `.0002`. After the pipeline runs, `PIPELINES_FILES` confirms only the targeted files were discovered.

In [15]:
%%sql
DROP PIPELINE IF EXISTS rtdm_pipeline_glob;

-- Pattern 000[0-2] matches files ending in .0000, .0001, or .0002 — a subset of the full dataset.
CREATE PIPELINE rtdm_pipeline_glob
AS LOAD DATA S3 'singlestore-realtime-digital-marketing/v2/1k-2p/purchases.*.000[0-2].json'
CONFIG '{"region": "us-east-1"}'
BATCH_INTERVAL 2500
INTO TABLE rtdm_purchases_raw
FORMAT JSON
(
    seq_id        <- seqId,
    subscriber_id <- subscriberid,
    @v            <- vendor
)
SET vendor = FROM_BASE64(@v);

START PIPELINE rtdm_pipeline_glob;

In [16]:
time.sleep(5)
try:
    run_sql("STOP PIPELINE rtdm_pipeline_glob")
except Exception:
    pass
print("Glob pipeline stopped. Checking discovered files...")

In [17]:
%%sql
SELECT FILE_NAME, FILE_SIZE, FILE_STATE
FROM   information_schema.PIPELINES_FILES
WHERE  PIPELINE_NAME = 'rtdm_pipeline_glob'
LIMIT  10;

## Section 7 — Cleanup

Stops and drops both pipelines, drops the working table and `pipelines_demo` database. Run this before rerunning the notebook from scratch.

In [18]:
for p in ["rtdm_pipeline", "rtdm_pipeline_glob"]:
    try:
        run_sql(f"STOP PIPELINE {p}")
    except Exception:
        pass

In [19]:
%%sql
DROP PIPELINE IF EXISTS rtdm_pipeline;
DROP PIPELINE IF EXISTS rtdm_pipeline_glob;
DROP TABLE IF EXISTS rtdm_purchases_raw;

In [20]:
run_sql(f"DROP DATABASE IF EXISTS {DB_NAME}")
print(f"Database '{DB_NAME}' dropped. Cleanup complete.")

## Key takeaways

<table>
  <tr><th>Concept</th><th>What this notebook showed</th></tr>
  <tr><td>Parquet complex types</td><td>Arrays and maps land as <code>JSON</code>; use <code>JSON_EXTRACT_STRING</code> / <code>JSON_ARRAY_UNPACK</code> to work with elements</td></tr>
  <tr><td><code>ALTER PIPELINE</code></td><td>Update credentials or bucket config on a live pipeline without resetting offsets — stop first, alter, restart</td></tr>
  <tr><td><code>PIPELINES_BATCHES</code></td><td>Per-batch timing and row counts; use <code>BATCH_TIME</code> vs <code>BATCH_ROWS_INSERTED</code> to find bottlenecks</td></tr>
  <tr><td>Event-driven discovery</td><td>Kinesis stream integration eliminates polling latency; file discovery time drops to near-zero</td></tr>
  <tr><td>Glob patterns</td><td>Use <code>?</code>, <code>[a-z]</code>, <code>[^x]</code> in S3 paths to reduce file list size and speed up discovery scans</td></tr>
</table>

<div id="singlestore-footer" style="background-color: rgba(194, 193, 199, 0.25); height:2px; margin-bottom:10px"></div>
<div><img src="https://raw.githubusercontent.com/singlestore-labs/spaces-notebooks/master/common/images/singlestore-logo-grey.png" style="padding: 0px; margin: 0px; height: 24px"/></div>